In [1]:
import pydrake.math
from pydrake.all import (
    StartMeshcat,
    RigidTransform,
    RotationMatrix,
    MinimumDistanceLowerBoundConstraint,
    cos, 
    sin, 
    sqrt,
    arcsin, 
    arccos,
    atan2,
    Rgba,
    Sphere,
    Cylinder,
    AutoDiffXd,
    RigidTransform_,
    RollPitchYaw_,
)
import sys, time
import os
parent_dir = os.path.abspath(os.path.join(os.curdir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from src.utils import RepoDir, BuildEnv, DrawAxes, DrawSphere, DrawCylinder
import numpy as np
np.set_printoptions(precision=9, suppress=True)

In [2]:
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat, os.path.join(RepoDir(), "models/panda/panda_finray_collision.yaml"))
plant = diagram.GetSubsystemByName("plant")
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [3]:
q = np.full(7, -1.)
# q[7:] = np.zeros(2)
q[5] = 1
plant.SetPositions(plant_context, q)
diagram.ForcedPublish(diagram_context)
plant.GetDefaultPositions()

EE = plant.GetFrameByName("between_fingers")
target = EE.CalcPoseInWorld(plant_context)

In [ ]:
# This used to be a verbatim copy of src/panda_analytic_ik.py, which drifted
# whenever the source changed (it missed the 8-branch elbow chart entirely).
# Import the real implementation instead.
from src.panda_analytic_ik import (
    Analytic_IK_Panda, panda_a, panda_d, panda_alpha,
    panda_limits_lower, panda_limits_upper,
    scalar_clip, safe_arccos, safe_arcsin, safe_norm, safe_divide,
)


In [ ]:
analytic_offset = RigidTransform(
        RotationMatrix([[0, 0., 1.], 
                        [0, -1, 0.],
                        [1., 0, 0.]]),
        np.array([-0.0236, -1.87933e-05, 0.0]),
    )

In [20]:
q = np.array([-1.5, -0.5, 1.5, -1.5, 0.5, 1.5, 0.])

plant.SetPositions(plant_context, q)
diagram.ForcedPublish(diagram_context)



ee_target = RigidTransform(ik.FK(q[:7]))
print(analytic_offset @ ee_target )

target = EE.CalcPoseInWorld(plant_context)
print(target)
DrawAxes(target, meshcat)

print(target.inverse().multiply(ee_target))



psi = ik.psi(q[:7])

gc = ik.gc(q[:7])
print(ik.IK(target, psi,  GC=gc, pose_offset=analytic_offset))
print(q)

q_sol = np.zeros(7)

q_sol[:7] = ik.IK(target, psi, GC=gc, pose_offset=analytic_offset)
print(gc)
plant.SetPositions(plant_context, q_sol)
diagram.ForcedPublish(diagram_context)



RigidTransform(
  R=RotationMatrix([
    [0.017044502146680105, 0.018670998440872378, -0.999680388306079],
    [-0.7365508657085164, 0.6763821528228037, 7.461115567833662e-05],
    [0.6761673662419734, 0.7363141837286651, 0.025280737121031342],
  ]),
  p=[0.4902527783377249, -0.2111816366703323, 0.5382033609951324],
)
RigidTransform(
  R=RotationMatrix([
    [0.02586707576050265, -0.7362938185392007, 0.6761673662434199],
    [-0.0006132323167264835, 0.6763818789500109, 0.7365508657073039],
    [-0.9996652031243891, -0.01946706472521093, 0.017044502141696883],
  ]),
  p=[0.5387999863906525, 0.21116108253506885, 0.4902603211739836],
)
RigidTransform(
  R=RotationMatrix([
    [-5.019716349198234e-12, 0.000796326719680182, 0.9999996829318275],
    [1.7882117593206616e-12, -0.9999996829318275, 0.000796326719680201],
    [1.0000000000000002, 1.792174487547009e-12, 5.018383962094173e-12],
  ]),
  p=[-0.023599992516897116, -1.8793308850995232e-05, 2.5510233656170893e-12],
)
[-1.499183618 -0.50

In [ ]:
for psi in np.linspace(-0.38, 1.0, 1000):
    q_sol = np.zeros(9)
    q_sol[:7] = ik.IK(target, psi, GC=gc)
    plant.SetPositions(plant_context, q_sol)
    diagram.ForcedPublish(diagram_context)
    time.sleep(0.005)
    

In [ ]:
psi

1.0